# Customer-safe split

Exported from the Split choice node. This notebook is plain Python and does not require Sprisk.

In [ ]:
import polars as pl

data = pl.DataFrame({
    "customer_id": ["a", "a", "b", "b", "c", "c", "d", "d", "e", "e"],
    "value": [11, 12, 20, 21, 31, 29, 40, 41, 51, 50],
})

In [ ]:
# Resolved from the Split choice node.
# Decision: group
# Reason: Keep every customer_id value entirely inside train or test.
split_choice = "auto"


def group_split(
    data: pl.DataFrame,
    group_column: str,
    test_fraction: float = 0.20,
    seed: int = 42,
):
    test_groups = (
        data.select(group_column)
        .unique()
        .sample(fraction=test_fraction, seed=seed, shuffle=True)
        .get_column(group_column)
    )
    train = data.filter(~pl.col(group_column).is_in(test_groups))
    test = data.filter(pl.col(group_column).is_in(test_groups))
    return train, test


train, test = group_split(
    data,
    group_column="customer_id",
    test_fraction=0.20,
    seed=42,
)

In [ ]:
assert set(train["customer_id"]).isdisjoint(set(test["customer_id"]))
train, test